In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [8]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
from email.mime import text
import json
from pyexpat.errors import messages


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [10]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [11]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{test_case['task']}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output  

In [12]:
def run_test_case(test_case):
    """Calls run_test_case, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [13]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [14]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(uri: str) -> Optional[str]:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI.\n    \n    Supports multiple S3 URI formats:\n    - s3://bucket-name/key\n    - s3a://bucket-name/key\n    - https://bucket-name.s3.region.amazonaws.com/key\n    - https://s3.region.amazonaws.com/bucket-name/key\n    - https://bucket-name.s3-region.amazonaws.com/key\n    \n    Args:\n        uri: S3 URI string\n        \n    Returns:\n        Region code (e.g., 'us-east-1') or None if not found\n    \"\"\"\n    if not uri:\n        return None\n    \n    # List of valid AWS regions\n    valid_regions = {\n        'us-east-1', 'us-east-2', 'us-west-1', 'us-west-2',\n        'eu-west-1', 'eu-west-2', 'eu-west-3', 'eu-central-1', 'eu-north-1',\n        'ap-southeast-1', 'ap-southeast-2', 'ap-northeast-